# CTM Robustness Check — WildChat Demographics Analysis

This notebook runs CombinedTM (Contextualized Topic Model) on the same preprocessed corpus used for BERTopic, as a robustness check.

**Reference:** Bianchi et al. (2021). Pre-training is a Hot Topic: Contextualized Document Embeddings Improve Topic Coherence. ACL 2021. https://aclanthology.org/2021.acl-short.96

**Outputs:**
- `results/ctm_topic_words.csv` — top words per CTM topic
- `results/ctm_vs_bertopic_comparison.csv` — side-by-side coherence/diversity comparison
- `results/ctm_state_topic_proportions.parquet` — state-level CTM topic proportions

## 0. Install dependencies
Run once, then restart kernel.

In [1]:
# Run once, then restart kernel
!pip install contextualized-topic-models sentence-transformers gensim pyarrow

## 1. Imports & setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from contextualized_topic_models.models.ctm import CombinedTM
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.evaluation.measures import CoherenceCV, TopicDiversity

Path("results").mkdir(exist_ok=True)

print("All imports OK")

All imports OK


## 2. Load data

In [2]:
df = pd.read_parquet("../../data/wildchat_bertopic.parquet")

print(f"Rows: {len(df):,}")
print(f"States: {df['state'].nunique()}")
print(f"Columns: {df.columns.tolist()}")

docs = df['user_text'].tolist()

Rows: 68,668
States: 36
Columns: ['conversation_hash', 'model', 'timestamp', 'hashed_ip', 'user_text', 'state']


## 3. Prepare data for CTM

CTM requires two inputs:
- **Bag-of-words** representation (sparse, vocabulary-based)
- **Contextual embeddings** (from a sentence transformer)

`TopicModelDataPreparation` handles both.

In [3]:
# This step computes both BoW and SBERT embeddings
# It will take ~10-20 minutes on CPU (similar to BERTopic embeddings)
# Uses all-MiniLM-L6-v2 to match BERTopic for fair comparison

qt = TopicModelDataPreparation("sentence-transformers/all-MiniLM-L6-v2")

print("Preparing training data (computing BoW + embeddings)...")
print("This will take ~10-20 minutes on CPU.")

training_dataset = qt.fit(text_for_contextual=docs, text_for_bow=docs)

print(f"\nVocabulary size: {len(qt.vocab):,}")
print(f"Training samples: {len(training_dataset)}")

Preparing training data (computing BoW + embeddings)...
This will take ~10-20 minutes on CPU.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/344 [00:00<?, ?it/s]


Vocabulary size: 105,019
Training samples: 68668


## 4. Run CombinedTM

We run CTM with the same number of topics as our best BERTopic solution (76) for a direct comparison.
We also run with 50 topics as an alternative.

CTM is a variational autoencoder — it trains over multiple epochs, which is why it's slower.

In [4]:
# Match BERTopic's best topic count for direct comparison
N_TOPICS_PRIMARY = 76
N_TOPICS_ALT = 50

print(f"Training CombinedTM with {N_TOPICS_PRIMARY} topics...")
print("This will run for 100 epochs — expect 1-3 hours on CPU.")

ctm = CombinedTM(
    bow_size=len(qt.vocab),
    contextual_size=384,        # all-MiniLM-L6-v2 output dimension
    n_components=N_TOPICS_PRIMARY,
    num_epochs=100,
    hidden_sizes=(100, 100),    # standard CTM architecture
    batch_size=64,
    lr=2e-3,
)

ctm.fit(training_dataset)
print("\nCTM training complete!")

Training CombinedTM with 76 topics...
This will run for 100 epochs — expect 1-3 hours on CPU.


Epoch: [100/100]	 Seen Samples: [6860800/6866800]	Train Loss: 777.5291348357699	Time: 0:06:33.508266: : 100it [11:02:27, 397.47s/it]
100%|██████████| 1073/1073 [01:35<00:00, 11.28it/s]


CTM training complete!


## 5. Extract and save topic words

In [9]:
TOP_N = 10
topics = ctm.get_topics(TOP_N)

print(type(topics))
print(topics)

topic_rows = []
for topic_id, words in topics.items():
    topic_rows.append({
        'topic_id': topic_id,
        'top_words': ', '.join(words),
        'topic_label': ''
    })

topic_df = pd.DataFrame(topic_rows)
topic_df.to_csv("results/ctm_topic_words.csv", index=False)

print("CTM Topics:")
print(topic_df.to_string(index=False))

<class 'collections.defaultdict'>
defaultdict(<class 'list'>, {0: ['respects', 'fulfilled', 'ticking', 'evoking', 'revels', 'cushions', 'diving', 'swell', 'delicacy', 'entrusted'], 1: ['evoking', 'respects', 'fulfilled', 'entrusted', 'delicacy', 'cushions', 'expertly', 'reassures', 'ticking', 'diving'], 2: ['quotes', 'neural', 'product', 'categories', 'term', 'risk', 'processing', 'choices', 'cost', 'cycle'], 3: ['view', 'triplet', 'sound', 'dream', 'newest', 'daughters', 'alive', 'loudly', 'portrait', 'phoenix'], 4: ['year', 'than', 'points', 'old', 'every', 'age', 'scenario', 'cast', 'because', 'years'], 5: ['digital', 'based', 'data', 'such', 'deployment', 'network', 'scikit', 'used', 'provide', 'and'], 6: ['please', 'your', 'thank', 'person', 'you', 'cassy', 'good', 'most', 'have', 'any'], 7: ['job', 'work', 'experience', 'resume', 'working', 'team', 'professional', 'dental', 'needs', 'customer'], 8: ['stage', 'move', 'button', 'display', 'self', 'text', '360', 'height', 'tkinter',

## 6. Score CTM and compare with BERTopic

In [15]:
# Coherence
print("Computing CTM coherence (may take a few minutes)...")

# Tokenize docs for coherence scoring
tokenized_docs = [doc.lower().split() for doc in docs]

# Convert defaultdict to list of word lists
topics_list = list(topics.values())

cv = CoherenceCV(texts=tokenized_docs, topics=topics_list)
ctm_coherence = cv.score()

# Diversity (manual calculation)
all_words = [word for words in topics_list for word in words]
ctm_diversity = len(set(all_words)) / len(all_words)

print(f"CTM Coherence (c_v): {ctm_coherence:.4f}")
print(f"CTM Diversity      : {ctm_diversity:.4f}")

# Load BERTopic scores for comparison
bertopic_log = pd.read_csv("results/bertopic_experiment_log.csv")
best_bertopic = bertopic_log[bertopic_log['run_id'] == 'run_04'].iloc[0]

# Build comparison table
comparison = pd.DataFrame([
    {
        'model': 'BERTopic (run_04)',
        'n_topics': best_bertopic['n_topics'],
        'coherence_cv': best_bertopic['coherence_cv'],
        'diversity': best_bertopic['diversity'],
        'outlier_rate': best_bertopic['outlier_rate'],
        'notes': 'min_topic_size=100, n_neighbors=15'
    },
    {
        'model': 'CombinedTM',
        'n_topics': N_TOPICS_PRIMARY,
        'coherence_cv': round(ctm_coherence, 4),
        'diversity': round(ctm_diversity, 4),
        'outlier_rate': 'N/A',
        'notes': 'n_components=76, epochs=100'
    }
])

comparison.to_csv("results/ctm_vs_bertopic_comparison.csv", index=False)
print("\nComparison table:")
print(comparison.to_string(index=False))

Computing CTM coherence (may take a few minutes)...
CTM Coherence (c_v): 0.4821
CTM Diversity      : 0.6132

Comparison table:
            model  n_topics  coherence_cv  diversity outlier_rate                              notes
BERTopic (run_04)        76        0.5806     0.8408       0.4493 min_topic_size=100, n_neighbors=15
       CombinedTM        76        0.4821     0.6132          N/A        n_components=76, epochs=100


## 7. State-level CTM topic proportions

In [16]:
# Get per-document topic distributions from CTM
# CTM returns a probability distribution over topics for each document
doc_topic_matrix = ctm.get_doc_topic_distribution(training_dataset, n_samples=20)

# doc_topic_matrix shape: (n_docs, n_topics)
print(f"Document-topic matrix shape: {doc_topic_matrix.shape}")

# Add to dataframe
topic_cols = [f"topic_{i}" for i in range(N_TOPICS_PRIMARY)]
df_ctm = df[['conversation_hash', 'state']].copy()
df_ctm[topic_cols] = doc_topic_matrix

# Aggregate to state level (mean topic proportion per state)
state_ctm_props = df_ctm.groupby('state')[topic_cols].mean().reset_index()

state_ctm_props.to_parquet("results/ctm_state_topic_proportions.parquet", index=False)
print(f"Saved ctm_state_topic_proportions.parquet")
print(f"Shape: {state_ctm_props.shape}")
print(state_ctm_props.head())

100%|██████████| 1073/1073 [02:14<00:00,  7.96it/s]


Document-topic matrix shape: (68668, 76)
Saved ctm_state_topic_proportions.parquet
Shape: (36, 77)
        state   topic_0   topic_1   topic_2   topic_3   topic_4   topic_5  \
0     Alabama  0.016877  0.016172  0.021196  0.010359  0.017095  0.010562   
1     Arizona  0.014070  0.014142  0.016174  0.008642  0.013162  0.017850   
2    Arkansas  0.013891  0.013758  0.013075  0.010263  0.011424  0.008821   
3  California  0.011151  0.011168  0.017164  0.007363  0.014845  0.016536   
4    Colorado  0.014200  0.014570  0.018025  0.009190  0.014069  0.011817   

    topic_6   topic_7   topic_8  ...  topic_66  topic_67  topic_68  topic_69  \
0  0.010155  0.013262  0.020031  ...  0.015407  0.011128  0.017172  0.008023   
1  0.009776  0.021750  0.016731  ...  0.013559  0.009784  0.013501  0.007143   
2  0.009316  0.011626  0.011049  ...  0.013664  0.010456  0.013583  0.008874   
3  0.009211  0.015346  0.016028  ...  0.011116  0.007823  0.011177  0.006430   
4  0.009744  0.017464  0.016263  ...  

## 8. Summary

In [17]:
print("=" * 50)
print("CTM ROBUSTNESS CHECK SUMMARY")
print("=" * 50)
print(f"Model          : CombinedTM (Bianchi et al., 2021)")
print(f"Topics         : {N_TOPICS_PRIMARY}")
print(f"Epochs         : 100")
print(f"Coherence (cv) : {ctm_coherence:.4f}")
print(f"Diversity      : {ctm_diversity:.4f}")
print(f"")
print("Output files:")
print("  results/ctm_topic_words.csv")
print("  results/ctm_vs_bertopic_comparison.csv  <-- key robustness table")
print("  results/ctm_state_topic_proportions.parquet")

CTM ROBUSTNESS CHECK SUMMARY
Model          : CombinedTM (Bianchi et al., 2021)
Topics         : 76
Epochs         : 100
Coherence (cv) : 0.4821
Diversity      : 0.6132

Output files:
  results/ctm_topic_words.csv
  results/ctm_vs_bertopic_comparison.csv  <-- key robustness table
  results/ctm_state_topic_proportions.parquet
